# Step 19.2 — Train / validation loop：沿着参数版本读懂循环

在仓库根目录同步母版：

```bash
git pull
```

再创建独立工作副本，保留已有笔记：

```bash
cp -n notebooks/83_step19_2_train_validation_loop.ipynb notebooks/work/83_step19_2_train_validation_loop_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上为未来使用的命令参考，本次未执行。环境仍是项目Python 3.12 / `.venv`、固定 **MLX v0.32.2**。全部代码单元未执行、输出为空；没有语料读取、模型初始化、训练、验证或测量结果。

2026-11-17 教学修订：**累计2/2＝历史依赖补齐1次＋本次教学整理1次**。没有重置原1/2计数。此次保留训练／验证接口与逻辑，去掉重复模型定义，增加版本时间线并将实际运行改为主动入口；只修改19.2。


## 阅读路线与本节新增内容

[19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/82_step19_1_hyperparameter_baseline.ipynb)给出配置候选；[17.5](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/76_step17_5_uncompiled_training_step.ipynb)与[18.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/80_step18_4_mutable_training_state.ipynb)已讲过单步更新、编译状态和求值。本节把这些连接为“更新若干次，定期在验证集上评分”的循环。

先读正文的时间线、模式／随机数边界及两个helper，再看附录中的完整旧依赖。正文函数中用到的get_batch和模型均在本Notebook附录提供；Python定义函数时不调用它们，未来真正调用必须在所需定义齐备之后。

| 区域 | 阅读目的 | 未来执行的含义 |
|---|---|---|
| 正文：配置、loss、验证helper、循环 | 理解每条记录对应哪一版参数 | 只有常量、导入与定义，不自动运行循环 |
| 附录A／B：旧数据与模型依赖 | 恢复一套自包含前置，可按链接回顾 | 定义不会读取缓存、采样或创建模型 |
| 附录C：匹配setup与注释调用 | 明确怎样主动准备、初始化、运行 | 调用才会有相应作用，不能当成无副作用的说明文本 |

本节保留原基础循环；下一节再讲稳定性检查，不把尚未教的裁剪或其他训练机制提前混入。


## 1. 先固定小计划，再谈循环

候选仍是C=64、N=2、H=4、T=64、B=4、学习率3e-4；模型位置容量max_seq_len从block_size映射。真实V必须来自同一份字符表，不能填一个猜测的词表大小。

原计划保持：**4次训练更新，每完成2次更新验证一次，每次验证抽2个batch**。此外，循环开始前先验证一次；最后一次更新也会验证，即使它没有落在整除周期上。这个4/2/2计划是便于读流程的候选，不是足够训练或评测模型的声明。

三种随机种子职责分开：model用于初始化，train用于持续训练采样，validation用于重复相同的验证采样安排。下面仅保存Python字典，不创建随机数生成器、数组或模型。


In [ ]:
baseline_candidate = {"model_width":64, "layers":2, "heads":4, "block_size":64, "batch_size":4, "learning_rate":3e-4}
loop_plan = {"num_updates":4, "eval_every":2, "val_batches":2}
seeds = {"model":2028, "train":2026, "validation":2027}


## 2. 一次更新，存在两个参数版本

用θ[k−1]表示第k次更新之前的模型参数，Dₖ表示该步从训练split抽到的(X,Y)。训练图先算scalar loss和梯度，再让Adam更新：

$$
L_k=L(\theta_{k-1};D_k),\qquad
g_k=\nabla_\theta L(\theta_{k-1};D_k),
$$
$$
(\theta_k,S_k)=\operatorname{AdamUpdate}(\theta_{k-1},S_{k-1},g_k).
$$

S表示Adam的状态，包括step与一阶／二阶矩。训练步返回的数组仍代表 **Lₖ：更新前参数的loss**。把模型引用更新为θ[k]，不会自动用新参数重新算这份loss；之后才调用`.item()`也不会改变它的含义。

循环随后 `mx.eval(train_loss_before_update, training_state)`，要求该loss以及**当前**模型／Adam状态完成，再做记录或验证。只读loss不能当作已明确完成所有新状态的边界。这个eval是数组求值，不是切换模型模式。

### 把原4/2/2计划放在一条时间线上

下面假设使用全新setup，初始Adam step=0。表中只有参数版本和事件，没有虚构loss数值：

| 完成更新数 | 训练步使用／返回的loss | 更新及求值后模型 | 此时验证使用什么？ |
|---:|---|---|---|
| 0 | 没有训练步，train字段为None | θ[0] | 初始验证：固定θ[0]，抽2批 |
| 1 | L_train(θ[0]; D₁) | θ[1] | 不验证，validation字段为None |
| 2 | L_train(θ[1]; D₂) | θ[2] | 周期验证：固定θ[2]，抽2批 |
| 3 | L_train(θ[2]; D₃) | θ[3] | 不验证 |
| 4 | L_train(θ[3]; D₄) | θ[4] | 周期／末次验证：固定θ[4]，抽2批 |

同一条完成数为2的记录里，train loss对应θ[1]，validation loss对应θ[2]；两列还来自不同数据。不能把两者当成同一参数、同一批输入的前后差。

函数实际从Adam当前step读取起点，不硬编码0。以后继续调用同一匹配setup时，记录沿当前完成数延续；周期条件使用这个全局完成数。计数每次加1的前提是传入的train_step确实进行**一次**Adam更新，不能换成只forward或更新两次的callable还沿用标签。


## 3. 训练与验证共用loss，但不共用更新路径

[17.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/73_step17_2_flatten_bt_supervision.ipynb)的契约保持：X/Y为(B,T)，Y已经由sampler右移一位。MiniGPT输出raw logits (B,T,V)，展平为(B×T,V)和(B×T)，逐位置cross-entropy后mean得到标量()。这里不再shift，也不先softmax后喂给交叉熵。

下面是已讲loss的复用。训练通过 `nn.value_and_grad(model, loss_fn)` 得到它的值和梯度；验证直接调用 `loss_fn(model, Xv, Yv)`，不调用梯度变换或optimizer。它们评分公式相同，职责不同。


In [ ]:
import random
import mlx.core as mx
import mlx.nn as nn

def loss_fn(model, X, Y):
    # X, Y: (B, T); Y is already shifted by get_batch.
    logits = model(X)  # (B, T, V), raw unnormalized logits
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))  # (B*T, V)
    flat_targets = mx.reshape(Y, (B * T,))        # (B*T,)
    token_losses = nn.losses.cross_entropy(
        flat_logits, flat_targets, axis=-1, reduction="none"
    )  # (B*T,)
    return mx.mean(token_losses)


### Train/eval mode与mx.eval：三个不同动作

| 调用 | 改变什么 | 不负责什么 |
|---|---|---|
| `model.train()` | 将模型及子模块设为training mode | 不自动取batch或更新权重 |
| `model.eval()` | 等价于`model.train(False)`，设为evaluation mode | 不自动关闭梯度、不冻结参数、不完成数组计算 |
| `mx.eval(...)` | 完成所选数组及其依赖 | 不改变模型train/eval模式 |

当前MiniGPT没有Dropout、BatchNorm或读取training标志的前向分支；同一参数和输入下，单纯切模式不改变其前向数学定义。仍保留模式切换，是为了让“训练阶段／验证阶段”的边界明确。以后换入依赖模式的层时，还须核对相应编译和状态行为，不能直接套用当前结论。

验证前记住 `was_training`，临时切eval，最后恢复原标志。`try ... finally` 的作用是：正常返回或循环内报错，都执行模式恢复，避免一次验证让后续阶段悄悄留在错误模式。验证不更新参数，来自代码里没有梯度和optimizer路径，不是来自eval这个名字。


## 4. 数据与随机数边界：重复比较同一组验证窗口

先把连续token stream做90/10顺序划分，再分别把train_tokens、val_tokens交给sampler。每个窗口从所选split内取T+1项，构成X=前T项、Y=后T项；不能先跨边界取窗口再标train/validation。本候选两边都至少要65个token，缺失或太短就停止，不用toy数据伪装真实语料。

| RNG对象 | 创建时机 | 状态怎样走 |
|---|---|---|
| 持久train_rng | 新setup创建一次 | 每步抽样继续消耗同一个对象，不能每步重置seed |
| 验证局部rng | 每次estimate_validation_loss调用时按val_seed重建 | 本次验证的多个batch依次消耗它，函数结束后丢弃 |

在val_tokens内容／长度、B、T与num_batches相同的条件下，每次验证重建同一种子，会复用相同的窗口起点安排，让比较更聚焦于参数变化；它不消耗train_rng。窗口可能重复或重叠，因此这是抽样估计，不是遍历完整验证集，也不是每次都得到新的独立评测数据。

### 两批mean怎样合成一个mean？

第j批平均loss为Lⱼ、监督位置数为nⱼ，应合成 `sum(nⱼ × Lⱼ) / sum(nⱼ)`。代码先把每批mean乘回位置数，再累计总位置数。当前每批nⱼ=B×T=256，故两批的等权平均恰好等于位置加权平均；若假设两批mean为1.2和1.8，纸算得到1.5。这是讲解用假设，不是本模型结果。

验证期间模型参数固定，`mx.eval(loss)`完成本批评分，`.item()`取得Python标量，才能累加到Python记录。以下完整helper对应刚才的模式恢复、独立RNG、批次加权三个步骤。


In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches < 1:
        raise ValueError('num_batches must be positive int')
    rng = random.Random(seed)
    was_training = model.training
    model.eval()
    try:
        total_loss = 0.0
        total_positions = 0
        for _ in range(num_batches):
            Xv, Yv, _ = get_batch(val_tokens, batch_size, block_size, rng)
            loss = loss_fn(model, Xv, Yv)
            mx.eval(loss)
            n = Yv.shape[0] * Yv.shape[1]
            total_loss += loss.item() * n
            total_positions += n
        return total_loss / total_positions
    finally:
        model.train(was_training)


## 5. 组合循环：先看记录字段，再读实现

`run_training_with_validation`接收同一组model、optimizer、train_step、training_state和持久train_rng。它不是自己另建一份“评估模型”。每次验证直接使用传入的**当前model**；不要提前复制旧参数树或旧模型来替换它。

`training_state = [model.state, optimizer.state]`保存原状态容器的引用，编译更新与求值都沿用这组容器。它不是冻结的参数快照。不能创建了新model却继续传旧compiled callable或旧state容器；附录的setup把这些对象一起创建并返回。

| history字段 | 含义 |
|---|---|
| completed_updates | 初始Adam step加上已进行的单步更新数 |
| train_loss_before_update | 刚完成的训练步在旧参数下的loss；初始记录为None |
| validation_loss_current | 本次用当前参数估计的validation loss；未到验证点为None |

`None`表示这条记录没有做该项评分，不是loss=0。条件 `completed % eval_every == 0` 识别周期；`local_update == num_updates`确保末次也评分。二者用or连接，恰好重合时只验证一次。

训练图只负责forward/loss、梯度与一次Adam update，返回旧参数loss。采样、模式切换、Python日志、验证和显式求值都在图外。下面的函数只定义流程，不在本单元调用。


In [ ]:
def run_training_with_validation(model, optimizer, train_step, training_state,
                                 train_tokens, val_tokens, train_rng, batch_size, block_size,
                                 num_updates, eval_every, val_batches, val_seed):
    if type(num_updates) is not int or num_updates < 1 or type(eval_every) is not int or (eval_every < 1):
        raise ValueError('Invalid loop plan')
    model.train()
    mx.eval(training_state)
    completed = int(optimizer.state['step'].item())
    initial_val = estimate_validation_loss(model, val_tokens, batch_size, block_size, val_batches, val_seed)
    history = [{'completed_updates': completed, 'train_loss_before_update': None, 'validation_loss_current': initial_val}]
    for local_update in range(1, num_updates + 1):
        model.train()
        X, Y, _ = get_batch(train_tokens, batch_size, block_size, train_rng)
        train_loss_before_update = train_step(X, Y)
        mx.eval(train_loss_before_update, training_state)
        completed += 1
        train_value = train_loss_before_update.item()
        val_value = None
        if completed % eval_every == 0 or local_update == num_updates:
            val_value = estimate_validation_loss(model, val_tokens, batch_size, block_size, val_batches, val_seed)
        history.append({'completed_updates': completed, 'train_loss_before_update': train_value, 'validation_loss_current': val_value})
    return history


## 6. 正文检查点

读循环时按这条顺序定位：初始验证 → 训练采样 → 更新前loss／梯度 → 一次Adam更新 → loss与新状态求值 → 必要时用当前模型验证 → 写一条记录。验证函数结束时恢复原模式；下一训练步也明确调用model.train()。

本节没有训练日志、验证曲线或已收敛结论。保留当前简单循环，有限值检查、梯度范数与裁剪在19.3另讲；本次不提前加入新算法。

自查（不附答案）：

1. 从时间线选一条同时有train和validation的记录，为两列写上参数版本与数据来源。
2. 为什么验证重新建立自己的RNG，却不能每个训练步都重建train_rng？
3. 若传入的train_step仅计算loss、不更新Adam，哪条计数约定被破坏？

以下附录用于未来独立工作副本。理解正文时可以先不展开完整模型；若以后要调用循环，所需定义必须先齐备。


## 附录A · 旧数据依赖与主动读取入口

以下配置映射沿用 [19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/82_step19_1_hyperparameter_baseline.ipynb)，数据读取／划分／采样沿用 [16.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/67_step16_1_acquire_inspect_tiny_shakespeare.ipynb)、[16.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/68_step16_2_train_validation_split.ipynb)、[16.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/69_step16_3_batch_sampler.ipynb)。它们是复用依赖，保留原接口。Adam继续显式bias_correction=True、betas=[0.9,0.999]、eps=1e-8；学习率来自候选，不把课程选择误写成固定版本的默认值。

必要导入如下；定义、配置字典与注释本身不会读取语料。可选下载函数仅在未来显式要求时使用。


In [ ]:
import math
from pathlib import Path
from urllib.request import urlopen
import mlx.optimizers as optim


In [ ]:
def make_model_kwargs(c, vocab_size):
    return {'vocab_size': vocab_size, 'dims': c['model_width'], 'num_heads': c['heads'], 'num_blocks': c['layers'], 'max_seq_len': c['block_size']}

def make_sampler_kwargs(c):
    return {'batch_size': c['batch_size'], 'block_size': c['block_size']}

def make_optimizer_kwargs(c):
    return {'learning_rate': c['learning_rate'], 'betas': [0.9, 0.999], 'eps': 1e-08, 'bias_correction': True}


In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found. Set PROJECT_ROOT to the repository root.')

def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response:
        raw = response.read()
    tmp = path.with_suffix('.tmp')
    tmp.write_bytes(raw)
    tmp.replace(path)
    return path


`load_cached_character_corpus`返回真实字符映射和一维token stream；`split_train_validation`按同一stream顺序切分。下面的定义不访问文件，调用loader才读取路径；缺失缓存会报错，不能以初始化V或占位文本继续冒充准备完成。


In [ ]:
def load_cached_character_corpus(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text = path.read_text(encoding='utf-8')
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return (text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32))

def split_train_validation(stream, fraction=0.9):
    if len(stream.shape) != 1 or stream.shape[0] < 2 or (not 0 < fraction < 1):
        raise ValueError('Expected 1-D stream and 0<fraction<1')
    s = int(stream.shape[0] * fraction)
    if s <= 0 or s >= stream.shape[0]:
        raise ValueError('Empty split')
    return (stream[:s], stream[s:], s)


`get_batch`仍只接受已经选好的一个split。它返回X、Y和窗口起点；X/Y都为(B,T)，起点由传入的RNG决定。训练与验证分别传各自split和RNG，边界由这些参数明确表达。


In [ ]:
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape) != 1 or type(batch_size) is not int or batch_size < 1 or (type(block_size) is not int) or (block_size < 1):
        raise ValueError('Invalid sampler inputs')
    n = data.shape[0]
    if n < block_size + 1:
        raise ValueError('Split needs block_size+1 tokens')
    starts = [rng.randrange(0, n - block_size) for _ in range(batch_size)]
    xs = []
    ys = []
    for s in starts:
        chunk = data[s:s + block_size + 1]
        xs.append(chunk[:-1])
        ys.append(chunk[1:])
    return (mx.stack(xs, axis=0), mx.stack(ys, axis=0), starts)


### 数据准备只在主动调用时发生

原稿的顶层bootstrap会在顺序运行时直接读取缓存。这里收进 `prepare_cached_data(project_root, config, *, acquire_missing_data=False)`：默认不下载；调用者指定仓库位置和实际候选。函数读取 `data/tinyshakespeare/input.txt`、取得真实V、做90/10划分与两边T+1检查，再显式求值token数组。

返回字典保存原数据变量；它不修改隐藏的全局变量，不创建模型、不抽batch。只有调用这一个入口时才发生读取和数组准备；后面的注释示例展示如何接回字段。


In [ ]:
def prepare_cached_data(project_root, config, *, acquire_missing_data=False):
    root = find_project_root(project_root)
    cache_path = root / "data" / "tinyshakespeare" / "input.txt"
    if acquire_missing_data:
        acquire_tiny_shakespeare(cache_path)
    text, characters, stoi, itos, token_stream = load_cached_character_corpus(cache_path)
    vocab_size = len(characters)
    train_tokens, val_tokens, split_index = split_train_validation(token_stream, 0.90)
    required_tokens = config["block_size"] + 1
    for split_name, split_data in (("train", train_tokens), ("validation", val_tokens)):
        if split_data.shape[0] < required_tokens:
            raise ValueError(f"{split_name} split has {split_data.shape[0]} tokens; need at least {required_tokens}.")
    mx.eval(train_tokens, val_tokens)
    return {
        "PROJECT_ROOT": root, "CACHE_PATH": cache_path,
        "text": text, "characters": characters, "stoi": stoi, "itos": itos,
        "token_stream": token_stream, "vocab_size": vocab_size,
        "train_tokens": train_tokens, "val_tokens": val_tokens, "split_index": split_index,
    }


## 附录B · 完整旧模型，仅保留一套

四个类均取原文件**最后生效的定义**，静态比对与 [15.4完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/65_step15_4_implement_minigpt.ipynb)一致。原403行块中三个底层类各重复三次；这些重复没有增加模型层数，现各留一份。没有改bias、Pre-Norm RMSNorm、因果mask、C→4C→C的MLP、独立Block、final RMSNorm或untied head。

下面较长代码是可选的前置恢复，不是19.2新知识。CausalMultiHeadAttention接收/返回(B,T,C)，内部score为(B,H,T,T)，形状细节见 [13.3拆头](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/54_step13_3_split_channels_into_heads.ipynb)；只定义类，不构造实例。


In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)


### MLP与Block

[14.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/60_step14_4_feed_forward_mlp.ipynb)的带bias两层MLP与GELU、两条Pre-Norm残差路径照旧。Block维持(B,T,C)；真正有几层，由MiniGPT的num_blocks决定，不由文本里重复写多少次class决定。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


### MiniGPT完整汇总定义

整数IDs (B,T)经token/position embedding和N个Block，输出raw logits (B,T,V)。T不能超出位置容量，V要匹配真实字符表。本段保留原来的完整检查和参数结构；不会在类定义之后自动新建模型。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


## 附录C · 匹配setup与可选未来运行

[18.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dfb8ec7ed5ddf991bfc092812df00fd655252657/notebooks/80_step18_4_mutable_training_state.ipynb)要求把同一个model、求导函数、Adam和原状态容器配套使用。下面把原setup收进 `make_training_setup`，只有主动调用才初始化这些对象。调用前已完成本Notebook的依赖定义，并取得真实vocab_size。

内部 `train_graph(X,Y)` 是一个**闭包（closure）**：它使用本次setup中的model、optimizer和loss_and_grad_fn，不依赖另一次运行留下的同名全局变量。返回的compiled_train_step仍是同样的(X,Y)接口；图内依旧只算loss／梯度、更新一次Adam并返回更新前loss。

`mx.compile(..., inputs=training_state, outputs=training_state, shapeless=False)` 使用同一组原状态容器。创建compiled callable不等于已经执行训练；图外的采样、模式切换与日志不移进去。编译不保证加速或降低内存。setup返回一个字典，将必须匹配的对象放在一起；重建时应整组替换，不能只换其中一个。


In [ ]:
def make_training_setup(config, vocab_size, seed_config):
    model_kwargs = make_model_kwargs(config, vocab_size=vocab_size)
    sampler_kwargs = make_sampler_kwargs(config)
    adam_kwargs = make_optimizer_kwargs(config)
    mx.random.seed(seed_config["model"])
    model = MiniGPT(**model_kwargs)
    model.train()
    loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
    optimizer = optim.Adam(**adam_kwargs)
    optimizer.init(model.trainable_parameters())
    training_state = [model.state, optimizer.state]
    train_rng = random.Random(seed_config["train"])
    mx.eval(training_state)

    def train_graph(X, Y):
        loss_before_update, grads = loss_and_grad_fn(model, X, Y)
        optimizer.update(model, grads)
        return loss_before_update

    compiled_train_step = mx.compile(
        train_graph, inputs=training_state, outputs=training_state, shapeless=False
    )
    return {
        "model": model, "optimizer": optimizer, "training_state": training_state,
        "train_rng": train_rng, "compiled_train_step": compiled_train_step,
        "sampler_kwargs": sampler_kwargs,
    }


### 三个不同的主动动作：准备数据、初始化、运行循环

以下全部是注释，因此Run All不会读取语料、初始化模型或启动训练。以后在工作副本确实选择操作时，按顺序逐段使用：第一段可能读文件／准备数组；第二段创建并求值模型和Adam初始状态；第三段才会真实训练和验证。不要把第三段视为“只是显示例子”。

数据准备和setup使用同一个selected_config；训练／验证用返回的两段真实tokens。调用循环时，从同一个setup字典取得model、optimizer、compiled callable和state；没有旧参数快照或另一份评估模型。


In [ ]:
# 1. 未来明确选择的数据准备；本次不取消注释、不执行。
# PROJECT_ROOT = Path("/absolute/path/to/mlx-minigpt-from-scratch")
# ACQUIRE_MISSING_DATA = False
# selected_config = baseline_candidate
# data = prepare_cached_data(PROJECT_ROOT, selected_config,
#                            acquire_missing_data=ACQUIRE_MISSING_DATA)
# train_tokens, val_tokens = data["train_tokens"], data["val_tokens"]
# vocab_size = data["vocab_size"]

# 2. 未来明确选择的新setup；不是一次训练更新。
# setup = make_training_setup(selected_config, vocab_size, seeds)
# sampler_kwargs = setup["sampler_kwargs"]

# 3. 未来主动调用；若取消注释，下面会真实训练与验证。
# history = run_training_with_validation(
#     model=setup["model"], optimizer=setup["optimizer"],
#     train_step=setup["compiled_train_step"], training_state=setup["training_state"],
#     train_tokens=train_tokens, val_tokens=val_tokens, train_rng=setup["train_rng"],
#     batch_size=sampler_kwargs["batch_size"], block_size=sampler_kwargs["block_size"],
#     num_updates=loop_plan["num_updates"], eval_every=loop_plan["eval_every"],
#     val_batches=loop_plan["val_batches"], val_seed=seeds["validation"],
# )
# for record in history:
#     print(record)


## 版本依据与材料状态

- [MLX v0.32.2 Module](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/base.py)：training标志、train/eval及state原容器引用。
- [v0.32.2 nn.value_and_grad](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)：对同一model的trainable parameters求标量loss梯度。
- [v0.32.2 transforms.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp)：compile显式状态输入/输出与eval接口。
- [v0.32.2 Optimizer／Adam](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)：状态与更新接口；保留课程显式Adam配置。

本次只整理19.2教学并静态核对JSON/schema、ID、语法、依赖与原实现。四个原模型类、loss、sampler、验证helper和训练循环的逻辑保持；原训练图移入匹配setup闭包，原数据/初始化/运行顶层动作改为显式入口和注释示例。

全部代码单元execution_count=null、outputs=[]。没有执行Notebook、配置/测试单元、训练、验证、实验、采样、测量、语料读取下载、checkpoint或用户工作副本。纸面假设不等于实测，材料不等于掌握。

19.2累计修正2/2（历史依赖修正1次，本次教学修订1次）；其他小节历史计数不变。本轮在19.2停止，等待独立验收。
